# AutoGen 基础示例

在这个代码示例中，你将使用 [AutoGen](https://aka.ms/ai-agents/autogen) AI 框架创建一个基本的智能体。

本示例的目标是向你展示我们稍后在实现不同智能体模式的附加代码示例中使用的步骤。

## 导入所需的 Python 包

In [ ]:
import os
from dotenv import load_dotenv

from autogen_agentchat.agents import AssistantAgent
from autogen_core.models import UserMessage
from autogen_ext.models.azure import AzureAIChatCompletionClient
from azure.core.credentials import AzureKeyCredential
from autogen_core import CancellationToken

from autogen_agentchat.messages import TextMessage
from autogen_agentchat.ui import Console


## 创建客户端

在本示例中，我们将使用 [GitHub Models](https://aka.ms/ai-agents-beginners/github-models) 来访问 LLM。

`model` 被定义为 `gpt-4o-mini`。尝试将模型更改为 GitHub Models 市场上可用的其他模型，以查看不同的结果。

作为快速测试，我们将运行一个简单的提示 - `What is the capital of France`。

In [ ]:
load_dotenv()
client = AzureAIChatCompletionClient(
    model="gpt-4o-mini",
    endpoint="https://models.inference.ai.azure.com",
    # To authenticate with the model you will need to generate a personal access token (PAT) in your GitHub settings.
    # Create your PAT token by following instructions here: https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens
    credential=AzureKeyCredential(os.getenv("GITHUB_TOKEN")),
    model_info={
        "json_output": True,
        "function_calling": True,
        "vision": True,
        "family": "unknown",
    },
)

result = await client.create([UserMessage(content="What is the capital of France?", source="user")])
print(result)

## 定义智能体

现在我们已经设置了 `client` 并确认它正在工作，让我们创建一个 `AssistantAgent`。每个智能体可以分配：
**name** - 一个简写名称，在多智能体流程中引用它时很有用。
**model_client** - 你在前面步骤中创建的客户端。
**tools** - 智能体可以用来完成任务的可用工具。
**system_message** - 定义 LLM 的任务、行为和语气的元提示。

你可以更改系统消息，看看 LLM 如何响应。我们将在第 4 课中介绍 `tools`。

In [ ]:
agent = AssistantAgent(
    name="assistant",
    model_client=client,
    tools=[],
    system_message="You are a travel agent that plans great vacations",
)

## 运行智能体

下面的函数将运行智能体。我们使用 `on_message` 方法用新消息更新智能体的状态。

在这种情况下，我们用来自用户的新消息 `"Plan me a great sunny vacation"` 更新状态。

你可以更改消息内容，看看 LLM 如何以不同的方式响应。

In [ ]:
from IPython.display import display, HTML


async def assistant_run():
    # Define the query
    user_query = "Plan me a great sunny vacation"

    # Start building HTML output
    html_output = "<div style='margin-bottom:10px'>"
    html_output += "<div style='font-weight:bold'>User:</div>"
    html_output += f"<div style='margin-left:20px'>{user_query}</div>"
    html_output += "</div>"

    # Execute the agent response
    response = await agent.on_messages(
        [TextMessage(content=user_query, source="user")],
        cancellation_token=CancellationToken(),
    )

    # Add agent response to HTML
    html_output += "<div style='margin-bottom:20px'>"
    html_output += "<div style='font-weight:bold'>Assistant:</div>"
    html_output += f"<div style='margin-left:20px; white-space:pre-wrap'>{response.chat_message.content}</div>"
    html_output += "</div>"

    # Display formatted HTML
    display(HTML(html_output))

# Run the function
await assistant_run()